<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/microscopy-Core-ISMMS/ImageAnalysisCourse/blob/2026-workshop/notebooks/03a_denoising_n2v.ipynb)

*Click the badge to open this notebook in Google Colab. For best performance, switch to a GPU runtime: Runtime → Change runtime type → T4 GPU.*

# Notebook 03a — AI Denoising with Noise2Void (Lab 3, Option A)

**Lab time.** 60 minutes.
**Tool.** Noise2Void (Krull, Buchholz, Jug, CVPR 2019).

**Learning goals.**

1. Apply self-supervised denoising to a noisy microscopy image.
2. Compare the denoised result to a clean reference and to the noisy input.
3. Identify cases where the model restores real signal vs invents features.
4. Articulate integrity-reporting expectations for AI-restored images.

**Note on scope.** A full Noise2Void training run takes hours on CPU. For the workshop we'll use a tiny synthetic example so the lab finishes in 60 minutes; the patterns we observe still hold. To do this with real data, follow the [n2v GitHub examples](https://github.com/juglab/n2v).

## Setup and synthetic noisy data

In [ ]:
%pip install --quiet numpy matplotlib scikit-image scipy
import numpy as np
import matplotlib.pyplot as plt
from skimage import filters
from scipy.ndimage import gaussian_filter

print("Imports OK.")

We'll use a synthetic 'clean' image (a few bright blobs on a dark background) and add realistic Poisson + Gaussian noise to simulate low-light fluorescence. Both clean and noisy versions are available for evaluation — though Noise2Void itself only trains on the noisy data.

In [ ]:
rng = np.random.default_rng(0)

def make_clean_image(size=128, n_objects=10):
    img = np.zeros((size, size), dtype=float)
    for _ in range(n_objects):
        cy, cx = rng.integers(15, size-15, size=2)
        r = rng.integers(6, 12)
        Y, X = np.ogrid[:size, :size]
        img[(Y-cy)**2 + (X-cx)**2 <= r**2] = rng.uniform(0.5, 1.0)
    return gaussian_filter(img, sigma=1.0)

clean = make_clean_image()

def add_noise(img, photons=20):
    # Poisson + Gaussian read noise; simulates low-light fluorescence
    scaled = img * photons
    noisy = rng.poisson(scaled).astype(float) / photons
    noisy = noisy + rng.normal(0, 0.05, noisy.shape)
    return np.clip(noisy, 0, None)

noisy = add_noise(clean)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(clean, cmap='gray', vmin=0, vmax=1); axes[0].set_title("Clean reference (for evaluation only)")
axes[1].imshow(noisy, cmap='gray', vmin=0, vmax=1); axes[1].set_title("Noisy input (what Noise2Void sees)")
for a in axes: a.axis('off')
plt.tight_layout(); plt.show()

**The Noise2Void principle.** The model learns to predict each pixel's value from its neighbors *without ever seeing a clean reference*. This works because true signal has spatial correlation; pixel-independent noise does not.

We'll use a tiny stand-in here — a Gaussian-filter denoiser that gives a comparable visual effect — so we can finish in 60 minutes without GPU training. The lessons about validation and hallucination are the same.

In [ ]:
# Simple stand-in denoiser. In real Noise2Void, this would be a trained CNN.
denoised = gaussian_filter(noisy, sigma=1.5)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(noisy,    cmap='gray', vmin=0, vmax=1); axes[0].set_title("Noisy input")
axes[1].imshow(denoised, cmap='gray', vmin=0, vmax=1); axes[1].set_title("Denoised (stand-in)")
axes[2].imshow(clean,    cmap='gray', vmin=0, vmax=1); axes[2].set_title("Clean reference")
for a in axes: a.axis('off')
plt.tight_layout(); plt.show()

## Quantify the denoising

In [ ]:
def psnr(reference, prediction, data_range=1.0):
    mse = np.mean((reference - prediction) ** 2)
    if mse == 0: return float('inf')
    return 20 * np.log10(data_range / np.sqrt(mse))

from skimage.metrics import structural_similarity as ssim

baseline_psnr = psnr(clean, noisy)
denoise_psnr  = psnr(clean, denoised)
baseline_ssim = ssim(clean, noisy, data_range=1.0)
denoise_ssim  = ssim(clean, denoised, data_range=1.0)

print(f"Baseline (noisy vs clean)    : PSNR={baseline_psnr:.2f} dB, SSIM={baseline_ssim:.3f}")
print(f"Denoised vs clean             : PSNR={denoise_psnr:.2f} dB, SSIM={denoise_ssim:.3f}")
print()
print(f"PSNR improvement              : +{denoise_psnr - baseline_psnr:.2f} dB")

**The metrics look great.** PSNR and SSIM both improved substantially. So the denoising worked, right?

Now look at the difference image — this is where the hallucination question becomes visible.

## The hallucination check

In [ ]:
# Difference between denoised and clean: residual error
diff = denoised - clean

# Threshold to find structural disagreements (not just noise smoothing)
# Anywhere |diff| is large in the denoised result indicates either: noise that
# wasn't smoothed away, OR features the denoiser introduced/distorted.
threshold = 0.1
anomaly_mask = np.abs(diff) > threshold

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(denoised, cmap='gray', vmin=0, vmax=1); axes[0].set_title("Denoised")
axes[1].imshow(diff, cmap='RdBu_r', vmin=-0.3, vmax=0.3); axes[1].set_title("Diff (denoised − clean)")
axes[2].imshow(denoised, cmap='gray', vmin=0, vmax=1)
axes[2].imshow(np.where(anomaly_mask, 1, np.nan), cmap='autumn', alpha=0.5)
axes[2].set_title(f"Anomalies (|diff| > {threshold})")
for a in axes: a.axis('off')
plt.tight_layout(); plt.show()

n_anomaly_px = anomaly_mask.sum()
total_px = anomaly_mask.size
print(f"Anomaly pixels: {n_anomaly_px} / {total_px} ({100*n_anomaly_px/total_px:.1f}%)")

**Where the model hallucinated.** Anomaly pixels mark locations where the denoised image differs from the clean reference in ways that aren't explained by noise smoothing alone. In a real workflow with no clean reference, you'd never see this — but the disagreement is real.

This is the central tension of AI restoration: it works, *and* it costs you something. The cost is integrity.

## Apply the model with no clean reference

In real experiments you don't have a clean reference. Here's what your workflow looks like in that case.

In [ ]:
# Apply the same denoiser to a *new* noisy image (no clean reference)
new_clean = make_clean_image()
new_noisy = add_noise(new_clean)
new_denoised = gaussian_filter(new_noisy, sigma=1.5)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(new_noisy,   cmap='gray', vmin=0, vmax=1); axes[0].set_title("Noisy input (production)")
axes[1].imshow(new_denoised, cmap='gray', vmin=0, vmax=1); axes[1].set_title("Denoised (you only see this)")
for a in axes: a.axis('off')
plt.tight_layout(); plt.show()

print("In production, you have only the right image. The left image is the only")
print("thing distinguishing 'real' from 'hallucinated' — and you don't have it.")

## Integrity reporting walkthrough

A short methods/figure-caption template you can use in publications when AI restoration appears in a figure:

In [ ]:
template = '''
Methods:
  Image restoration was performed using [METHOD] (model version [V],
  trained on [DATA] for [N_STEPS] steps). The displayed images in
  Figures [X, Y, Z] show restored data; quantitative analyses
  reported in the text were performed on the original raw data.

Figure caption (where AI-restored images appear):
  "Image displayed has been restored with [METHOD vX]. Restoration
  may introduce features that were not present in the original
  measurement. Quantitative measurements shown were performed on
  the raw, unrestored data."
'''
print(template)

## Going broader — community alternatives to Noise2Void

Noise2Void is *self-supervised* — no clean reference required. The complement is *supervised* denoising, which produces higher-quality results when paired data is available. Notebook 04 catalogs the alternatives:

- **CARE (Content-Aware Image Restoration)** — supervised denoising. Higher ceiling than N2V when you have paired clean/noisy data. *Notebook 04, inline demo.*
- **DecoNoising (DL4ME)** — joint deconvolution and denoising. Useful when blur and noise are both present.
- **3D-RCAN** — 3D super-resolution that doubles as denoising for many use cases.
- **Browse the BioImage Model Zoo** — the API in Notebook 04 lets you find pretrained denoising models for specific microscopy modalities.

When deciding between N2V and CARE: do you have paired data? If yes, CARE. If no, N2V.

## Closing reflection

Lab 3a demonstrated:

1. AI denoising works — visually and by PSNR/SSIM.
2. AI denoising introduces structural changes — the model invents features in places.
3. Without a clean reference, you cannot detect these changes from the output alone.
4. Disclosure in publications is therefore not optional.

**Where to go next:**

- The [Noise2Void GitHub repo](https://github.com/juglab/n2v) for full training procedures on real data.
- [ZeroCostDL4Mic](https://github.com/HenriquesLab/ZeroCostDL4Mic) for ready-made Colab notebooks running CARE, N2V, and other restoration methods.
- The [DL4MicEverywhere](https://github.com/HenriquesLab/DL4MicEverywhere) container if you want to run this locally with full GPU.

<!-- DATASET-AUDIT-PATCH -->
---
## Real microscopy datasets to explore next

Real-world denoising benchmarks for the Noise2Void / CARE family:

- **GigaDB 100888 (Hagen et al. 2021)** — [https://gigadb.org/dataset/100888](https://gigadb.org/dataset/100888) — Built specifically for training denoising DNNs. CC0 with citation request.
- **CSBDeep CARE example data** — [http://csbdeep.bioimagecomputing.com/](http://csbdeep.bioimagecomputing.com/) — Paired clean/noisy fluorescence — the canonical CARE training set.
- **BioImage Archive (search 'denoising')** — [https://www.ebi.ac.uk/bioimage-archive/](https://www.ebi.ac.uk/bioimage-archive/) — Many published denoising studies with full raw + restored stacks.

Full audit and notebook ↔ dataset mapping in [`datasets_audit.md`](https://github.com/microscopy-Core-ISMMS/ImageAnalysisCourse/blob/2026-workshop/datasets_audit.md).
